Step 1: Install Necessary Libraries
Install required Python libraries:

Pandas: for managing data efficiently using data frames.
NumPy: for powerful array operations and numerical tasks.
Matplotlib: for creating visualizations to gain insights from the data.
Scikit-learn: machine learning library

re ---
Python Regular Expressions module, which lets you search, match, and manipulate strings using patterns.

In [ ]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

Step 2: Load and Explore the Dataset
Dataset is in CSV format with columns review and sentiment. Use pandas to read the CSV file and inspect the structure, missing values and basic statistics.






In [ ]:
try:
    data = pd.read_csv('/content/sample_data/IMDB-Dataset.csv', encoding='utf-8')
except UnicodeDecodeError:
    data = pd.read_csv('/content/sample_data/IMDB-Dataset.csv', encoding='latin-1')
print(data.head())
data.head()
data.tail()

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


Step 3: Preprocess the Data
Here we will do:





1.   Lowercase conversion of text
2.   Removing HTML tags and special characters
3.   Remove extra spaces







In [ ]:
def clean_text(text):
    text = text.lower()
    #remove HTML tags
    text = re.sub('<.*?>', '', text)
    #remove special characters
    text = re.sub('[^a-zA-Z]', ' ', text)
     #remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
data['clean_review'] = data['review'].apply(clean_text)

Step 4: Convert Text to Numerical Vectors (TF-IDF)

*   Converts text into sequences of integers called vectors using tokenizer.

*   Pads sequences to make all reviews the same length (max_len=200) for LSTM input.




In [ ]:
#Prepare data for LSTM
X = data['clean_review']
y = data['sentiment'].map({'positive': 1, 'negative': 0})

#Tokenization
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len)

Step 5: Train-Test Split
Split the dataset into training and testing sets i.e 80 for training and 20% for testing to evaluate model performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Step 6: Build the model
Defines an LSTM neural network as:
*   Embedding Layer: Turns word indices into dense vectors.
*   LSTM Layer: Learns sequential patterns in text.
*   Dense Layer: Outputs the final sentiment prediction (0 or 1).



In [ ]:
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Step 7: Compile the Model
*   Specifies loss function like binary_crossentropy for binary classification.
*   Uses the Adam optimizer and tracks accuracy.





In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

Step 8: Train the model


*   Trains the model on the training data.
*   Runs for 5 epochs with a batch size of 64.
*   Uses 10% of the training data for validation during training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

Epoch 1/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 373s 654ms/step - accuracy: 0.7329 - loss: 0.5251 - val_accuracy: 0.8487 - val_loss: 0.3586
Epoch 2/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 366s 650ms/step - accuracy: 0.8647 - loss: 0.3327 - val_accuracy: 0.8702 - val_loss: 0.3203
Epoch 3/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 380s 648ms/step - accuracy: 0.8923 - loss: 0.2725 - val_accuracy: 0.7943 - val_loss: 0.4263
Epoch 4/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 362s 643ms/step - accuracy: 0.9078 - loss: 0.2392 - val_accuracy: 0.8602 - val_loss: 0.3742
Epoch 5/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 360s 640ms/step - accuracy: 0.9160 - loss: 0.2179 - val_accuracy: 0.8245 - val_loss: 0.3955


Step 9: Evaluate the model


*   Makes predictions on test data.
*   Converts probabilities to binary predictions (> 0.5 = positive).
*   Prints overall accuracy and detailed classification report (precision,    
    recall, F1-score).





In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

313/313 ━━━━━━━━━━━━━━━━━━━━ 23s 73ms/step
Accuracy: 0.8572
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      4961
           1       0.87      0.84      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



Step 10: Making predictions
*   This loop takes user input a movie review, cleans and tokenizes it
*   Then uses the trained LSTM model to predict whether the sentiment is
     positive or negative and displays the result. It runs until the user types 'exit'.




In [ ]:
while True:
    user_input = input("Enter a movie review (or type 'exit' to quit):\n")
    if user_input.lower() == 'exit':
        break

    # Clean and preprocess input
    cleaned_input = clean_text(user_input)
    input_seq = tokenizer.texts_to_sequences([cleaned_input])
    input_pad = pad_sequences(input_seq, maxlen=max_len)

    # Predict
    prediction = model.predict(input_pad)[0][0]
    sentiment = "Positive" if prediction > 0.5 else "Negative"

    print(f"\nPredicted Sentiment: {sentiment}")
    print(f"Confidence: {prediction:.2f}\n")

Enter a movie review (or type 'exit' to quit):
The movie is good
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step

Predicted Sentiment: Negative
Confidence: 0.46

Enter a movie review (or type 'exit' to quit):
Movie is great
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

Predicted Sentiment: Positive
Confidence: 0.58

Enter a movie review (or type 'exit' to quit):
exit
